# 04 — Full model fine-tuning

**Purpose:** Update all model parameters using the same examples, chat loss, optimizer family, and evaluation rules as LoRA.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# GitHub setup: run this as the FIRST code cell in Kaggle or Colab.
# This clones the source code only; GPU checkpoints/results need separate storage.
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
REPO_FOLDER = "nlp_research_proj"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # True: update an existing clean clone from GitHub

if Path("/kaggle/working").is_dir():
    workspace = Path("/kaggle/working")
elif Path("/content").is_dir():
    workspace = Path("/content")
else:
    workspace = Path.cwd()

repo_dir = workspace / REPO_FOLDER
project_dir = repo_dir / PROJECT_FOLDER

# Local development support: use the existing source folder outside Kaggle/Colab.
local_project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_utils.py").is_file()), None)
if local_project is not None and not (Path("/kaggle/working").is_dir() or Path("/content").is_dir()):
    ROOT = local_project
else:
    if not repo_dir.exists():
        print("Cloning:", REPO_URL)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    elif PULL_LATEST:
        # Only fast-forward; protects existing local changes from being overwritten.
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        print("Using existing clone:", repo_dir)

    ROOT = project_dir

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(
        f"Cannot find project_utils.py at {ROOT}. Confirm the GitHub folder name."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print("Project loaded from:", ROOT)


**IMPORTANT MEMORY NOTE:** Full training of 0.5B requires model weights, gradients, and optimizer state. Adafactor is selected **for both LoRA and full** to reduce optimizer-memory pressure and keep the optimizer family controlled. It is *not* guaranteed to fit T4 in every session. Start with the 2-step pilot in notebook 00 and batch size 1.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU'
TASK='sib200'; LANG='ur'; SIZES=[10]; SEEDS=[42]
MODEL_OVERRIDE=None # optionally an English-trained merged checkpoint, same as LoRA notebook
DRY_RUN=False
for seed in SEEDS:
    for n in SIZES:
        run_name=f'{TASK}_full_{LANG}_n{n}_s{seed}' + ('_anchor' if MODEL_OVERRIDE else '')
        if (ROOT/'outputs/checkpoints'/run_name/'complete.json').exists():
            print('Already complete:',run_name);continue
        if DRY_RUN: print('Would run',run_name); continue
        try:
            r=train_run('full',TASK,n,seed,run_name=run_name,lang=LANG,model_override=MODEL_OVERRIDE)
            print('FINISHED',r['run_id'],'minutes:',round(r['wall_seconds']/60,2),'GPU GiB:',r['peak_gpu_allocated_gib'])
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            print('FULL TRAIN OOM. Stop this kernel; reduce max_length to 128 or use more memory.')
            print('Do not claim a full-tuning result until a completed checkpoint is saved.')
            raise

When you have successful results at 10/class, add 25 and 50/class. Compare the same sizes and seeds as LoRA. Learning rates can differ between methods, but tune them fairly using the validation set, then freeze settings before final test evaluation.